In [1]:
# 1.0 EXPECTILE GAM PRIMER
# -----------------------------------------------------------------------------
# This section showcases how Expectile GAMs handle non-linear price elasticity 
# and varying levels of data noise (outliers).

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygam import s, ExpectileGAM


In [3]:

# 2.0 DATA GENERATION WITH OUTLIERS
# -----------------------------------------------------------------------------
np.random.seed(0)
n = 100

# Generate base price and quantity with a linear relationship + noise
price = np.sort(np.random.exponential(scale=100, size=n))
quantity = 1000 - 5 * price + np.random.normal(loc=0, scale=50, size=n)
quantity = quantity.clip(min=0) # Ensure quantity is never negative

# Add high-volume outliers (simulating promotional "spikes")
n_outliers = 10
outlier_prices = np.random.uniform(5, 50, n_outliers)
outlier_quantity = 1100 + np.random.normal(loc=0, scale=50, size=n_outliers)

price = np.concatenate([price, outlier_prices])
quantity = np.concatenate([quantity, outlier_quantity])

# Add low-volume outliers (simulating stock-outs or low demand periods)
n_outliers = 10
outlier_prices = np.random.uniform(51, 100, n_outliers)
outlier_quantity = 900 + np.random.normal(loc=0, scale=50, size=n_outliers)

price = np.concatenate([price, outlier_prices])
quantity = np.concatenate([quantity, outlier_quantity])

# Organize into a DataFrame and filter out extremely low prices
df = pd.DataFrame({'Price': price, 'Quantity': quantity})
df = df[df['Price'] >= 5]


In [ ]:

# 3.0 QUANTILE MODELING
# -----------------------------------------------------------------------------
# Prepare the data features (X) and target variable (y)
X = df[['Price']]
y = df['Quantity']

# Define quantiles: 2.5% (Lower Bound), 50% (Median), 97.5% (Upper Bound)
quantiles = [0.025, 0.5, 0.975]
gam_results = {}

for q in quantiles:
    # 's(0)' creates a smoothing spline for the Price column
    gam = ExpectileGAM(s(0), expectile=q)
    gam.fit(X, y)
    gam_results[q] = gam # Store the fitted model for each quantile

# 4.0 VISUALIZATION
# -----------------------------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.scatter(df['Price'], df['Quantity'], alpha=0.5, label='Data Points')

# Generate a smooth line for predictions across the price range
XX = np.linspace(df['Price'].min(), df['Price'].max(), 1000).reshape(-1, 1)

for q, gam in gam_results.items():
    # Plotting each quantile to show the "envelope" of demand
    plt.plot(XX, gam.predict(XX), label=f'{int(q*100)}th Quantile GAM')

plt.xlabel('Price')
plt.ylabel('Quantity Demanded')
plt.title('Quantile GAMS on Price Elasticity of Demand (Outliers Included)')
plt.legend()
plt.grid(True, linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show() #